#

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_pickle('combined_df_all_databases_only_english_3.pkl')



In [ ]:
df.shape

(15931, 7)

In [ ]:
df

,PubMed ID,DOI,Title,Year,Abstract,Author Keywords,english
0,16985399.0,10.1097/00001786-200610000-00006,Missed nursing care: a qualitative study,2006,The purpose of this study was to determine nur...,[],en
1,15946170.0,10.1111/j.1365-2934.2005.00541.x,Confirming mentorship,2005,Mentorship is related to nurses' success in nu...,[],en
2,17147757.0,10.1111/j.1741-6787.2004.04008.x,Methods in translation science,2004,The gap between research findings and practice...,[],en
3,19076465.0,10.1111/j.1744-6198.2008.00115.x,Competency: a concept analysis,2008,Concept analysis of competency.\nThe purpose o...,[],en
4,15946274.0,10.1111/j.1365-2702.2005.01136.x,Clinical education facilitators: a literature ...,2005,"The aim of this literature review, set within ...",[],en
...,...,...,...,...,...,...,...
15926,NaN,NaN,Enhancing Nursing Practice in Developmental Ca...,2008,This study aimed to enhance nursing practice i...,developmental care; participatory action resea...,en
15927,NaN,NaN,Project Evidencia [evidence] - research and ed...,2013,Project Evidencia [Evidence] intends to promot...,"Evidence-based nursing; Databases, bibliograph...",en
15928,NaN,NaN,Use of Research by Undergraduate Nursing Stude...,2017,Research utilization (RU) is crucial to prepar...,Nursing; Students; Nursing Research; Research ...,en
15929,NaN,NaN,Effects of an Eating Behaviour Modification Pr...,2013,The purpose of this study was to evaluate the ...,Eating behaviour; diabetes; obesity; modificat...,en


#inclusion_exclusion criteria

In [ ]:
inclusion_criteria = (
    "1. The article discusses barriers or facilitators to nurses' involvement in research activities.\n"
    "2. The article identifies challenges nurses face in conducting research, such as educational deficiencies, time constraints, organizational or cultural barriers, or lack of resources.\n"
    "3. The article proposes solutions or strategies to overcome barriers to nursing research participation.\n"
    "4. The article addresses the integration of research findings into clinical practice (Evidence-Based Practice).\n"
    "5. The article examines organizational cultures, policies, or leadership styles affecting nurses’ research involvement.\n"
)

exclusion_criteria = (

    "1. The article does not focus on nursing research or its associated barriers and solutions.\n"
    "2. The article lacks sufficient detail to evaluate against the inclusion criteria."
)

In [ ]:
prompt = """
        "You are a researcher rigorously screening titles and abstracts of scientific papers for inclusion or exclusion in a review paper. "
        "Use the criteria below to inform your decision. If any exclusion criteria are met or not all inclusion criteria are met, exclude the article. "
        "If all inclusion criteria are met, include the article. Only type 'included' or 'excluded' to indicate your decision. Do not type anything else.\n\n"
        f"Title: {title}\n"
        f"Abstract: {abstract}\n\n"
        f"Inclusion criteria: {inclusion_criteria}\n"
        f"Exclusion criteria: {exclusion_criteria}\n\n"
        "Decision: (here)"
    """


#Apply on all the data

In [ ]:
import json
import time
import os
import pandas as pd
from openai import OpenAI
from tqdm import tqdm

# Initialize OpenAI client
client = OpenAI(api_key="key_api")

# Save every X seconds
SAVE_INTERVAL = 120

# Paths
SAVE_PATH = "/content/drive/MyDrive/fix_systematic_review/output_review.xlsx"

# Load existing processed file if it exists
if os.path.exists(SAVE_PATH):
    print(f"🔄 Loading existing file from: {SAVE_PATH}")
    df = pd.read_excel(SAVE_PATH)
else:
    df = pd.read_pickle('combined_df_all_databases_only_english_3.pkl')
    # Create 'response' column if not present
    if 'response' not in df.columns:
        df['response'] = None

# Track time for periodic save
last_save_time = time.time()

# Iterate with progress bar
for index, row in tqdm(df.iterrows(), total=len(df), desc="Processing abstracts"):

    # Skip rows with existing responses
    if pd.notnull(row['response']):
        continue

    # Format the prompt
    full_prompt = prompt.format(
        inclusion_criteria=inclusion_criteria,
        exclusion_criteria=exclusion_criteria,
        title=row['Title'],
        abstract=row['Abstract']
    )

    try:
        # Call OpenAI API
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            temperature=0,
            max_tokens=2048,
            top_p=0.45,
            messages=[
                {"role": "user", "content": full_prompt}
            ]
        )

        # Extract and store response
        content = response.choices[0].message.content
        print(content)
        df.at[index, 'response'] = content

    except Exception as e:
        print(f"❌ Error at index {index}: {e}")
        continue

    # Save periodically
    current_time = time.time()
    if current_time - last_save_time > SAVE_INTERVAL:
        df.to_excel(SAVE_PATH, index=False)
        print(f"💾 Checkpoint saved at {SAVE_PATH}")
        last_save_time = current_time

# Final save
df.to_excel(SAVE_PATH, index=False)
print(f"✅ Final save completed at {SAVE_PATH}")


🔄 Loading existing file from: /content/drive/MyDrive/fix_systematic_review/output_review.xlsx


Processing abstracts:  14%|█▎        | 2174/15931 [00:00<00:00, 21738.00it/s]

excluded
excluded
excluded
excluded
excluded
excluded
excluded
excluded
excluded
excluded
excluded
excluded
excluded
excluded
excluded
excluded
excluded
excluded
excluded
excluded
excluded
excluded
excluded
excluded
excluded
excluded
excluded
excluded
excluded
excluded
excluded
excluded
excluded
excluded
excluded
excluded
excluded
excluded
excluded


Processing abstracts:  14%|█▎        | 2174/15931 [00:17<00:00, 21738.00it/s]

excluded


Processing abstracts:  18%|█▊        | 2831/15931 [00:17<01:44, 125.10it/s]  

... (25,332 outputs left out)

excluded


Processing abstracts: 100%|██████████| 15931/15931 [1:40:10<00:00,  2.65it/s]

excluded


✅ Final save completed at /content/drive/MyDrive/fix_systematic_review/output_review.xlsx


In [ ]:
import json
import time
import os
import pandas as pd
from openai import OpenAI
from tqdm import tqdm


# Paths
SAVE_PATH = "/content/drive/MyDrive/fix_systematic_review/output_review.xlsx"

# Load existing processed file if it exists
if os.path.exists(SAVE_PATH):
    print(f"🔄 Loading existing file from: {SAVE_PATH}")
    df = pd.read_excel(SAVE_PATH)


🔄 Loading existing file from: /content/drive/MyDrive/fix_systematic_review/output_review.xlsx


In [ ]:
df[df['response'] == 'excluded'].sample(25).to_excel('Excluded_25.xlsx')
df[df['response'] == 'included'].sample(25).to_excel('included_25.xlsx')